In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import kagglehub

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    PrecisionRecallDisplay,
    RocCurveDisplay,
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

In [ ]:
# Download latest version
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")

print("Path to dataset files:", path)
print(os.listdir(path))

In [ ]:
df = pd.read_csv(os.path.join(path, "creditcard.csv"))

print("Boyut:", df.shape)
print("Eksik değer:", int(df.isna().sum().sum()))
print("Tekrarlı satır:", int(df.duplicated().sum()))
df.head()

In [ ]:
counts = df["Class"].value_counts()
print(counts)
print(f"Fraud percent: {df['Class'].mean():.4%}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x="Class", data=df, ax=ax[0])
ax[0].set_yscale("log")
ax[0].set_title("Class distribution (log scale)")

sns.boxplot(x="Class", y="Amount", data=df, ax=ax[1], showfliers=False)
ax[1].set_title("Amount by Class")
plt.tight_layout()
plt.show()

In [ ]:
X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Train:", X_train.shape, f"fraud={y_train.mean():.4%}")
print("Test :", X_test.shape, f"fraud={y_test.mean():.4%}")

In [ ]:
preprocess = ColumnTransformer(
    [("scale", StandardScaler(), ["Time", "Amount"])],
    remainder="passthrough",
)

model = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(
        class_weight="balanced",
        solver="lbfgs",
        max_iter=2000,
        C=1.0,
        random_state=RANDOM_STATE,
    )),
])
model

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

oof_proba = cross_val_predict(
    model, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]

prec, rec, thr = precision_recall_curve(y_train, oof_proba)
f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
best_idx = int(np.argmax(f1))
threshold = float(thr[best_idx])

print(f"CV PR-AUC     : {average_precision_score(y_train, oof_proba):.4f}")
print(f"En iyi eşik   : {threshold:.4f}")
print(f"CV F1 (eşikte): {f1[best_idx]:.4f}")

plt.figure(figsize=(7, 4))
plt.plot(thr, prec[:-1], label="Precision")
plt.plot(thr, rec[:-1], label="Recall")
plt.plot(thr, f1, label="F1")
plt.axvline(threshold, color="k", ls="--", label=f"Seçilen eşik = {threshold:.3f}")
plt.xlabel("Eşik"); plt.legend(); plt.title("CV: eşik vs metrikler")
plt.show()

In [ ]:
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]

print(f"ROC-AUC: {roc_auc_score(y_test, proba):.4f}")
print(f"PR-AUC : {average_precision_score(y_test, proba):.4f}")

In [ ]:
for name, t in [("Varsayılan eşik 0.5", 0.5), ("Optimize eşik", threshold)]:
    pred = (proba >= t).astype(int)
    print(f"===== {name} ({t:.4f}) =====")
    print(classification_report(y_test, pred, digits=4))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, (name, t) in zip(ax, [("Eşik 0.5", 0.5), ("Optimize eşik", threshold)]):
    cm = confusion_matrix(y_test, (proba >= t).astype(int))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=a,
                xticklabels=["Normal", "Fraud"], yticklabels=["Normal", "Fraud"])
    a.set_title(f"{name} ({t:.3f})"); a.set_xlabel("Tahmin"); a.set_ylabel("Gerçek")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
PrecisionRecallDisplay.from_predictions(y_test, proba, ax=ax[0])
ax[0].set_title("Precision-Recall eğrisi")
RocCurveDisplay.from_predictions(y_test, proba, ax=ax[1])
ax[1].set_title("ROC eğrisi")
plt.tight_layout()
plt.show()

In [ ]:
feature_names = ["Time", "Amount"] + [c for c in X.columns if c not in ("Time", "Amount")]
coefs = pd.Series(model.named_steps["clf"].coef_[0], index=feature_names)
top = coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(15)

plt.figure(figsize=(7, 5))
top.sort_values().plot(kind="barh")
plt.title("The 15 most influential traits (logistic regression coefficient)")
plt.show()

In [ ]:
joblib.dump({"model": model, "threshold": threshold}, "fraud_logreg.joblib")

# Yükleme ve tahmin örneği
bundle = joblib.load("fraud_logreg.joblib")
sample = X_test.iloc[:5]
p = bundle["model"].predict_proba(sample)[:, 1]
print(pd.DataFrame({"fraud_proba": p, "tahmin": (p >= bundle["threshold"]).astype(int),
                    "gerçek": y_test.iloc[:5].values}))